# Trackastra - OFFLINE packaging validation (Phase 2 kill-criterion)

Go/no-go gate before investing in Phase 2 modeling. Answers 3 questions:
1. **Install offline?** `pip download` trackastra as wheels, then `pip install --no-index` with no network.
2. **Model offline?** Bundle a **pretrained** model and load it from disk. **Does a 3D model exist?** (our data is 3D+t; if only 2D pretrained ships, zero-shot is off the table -> Phase 2 = train Trackastra ourselves).
3. **Runs greedy (no ILP solver)?** `mode='greedy'` should not need an ILP solver (hard to package offline).

## Lesson from run 1 (why this version isolates the env)
Run 1 crashed: `pip install trackastra` **upgraded numpy to 2.4.6** inside the running Kaggle kernel, causing a numpy python/C ABI skew (`_blas_supports_fpe` AttributeError) when trackastra's import chain (dask->xarray->scipy) loaded. Fixes here: **(a) pin numpy/scipy to Kaggle's versions** so the resolver never upgrades them, and **(b) run all trackastra import/inference in a FRESH SUBPROCESS** so no stale in-kernel numpy is involved.

## Two-phase workflow (Kaggle cannot toggle internet mid-run)
- **Phase 1 - Internet ON:** `PHASE='download'`, Run All -> wheels + pretrained model into `/kaggle/working`. Then **Save Version** and make a Kaggle **dataset** from it (e.g. `trackastra-offline`).
- **Phase 2 - Internet OFF:** `PHASE='offline_test'`, **attach that dataset**, Internet **OFF**, Run All -> offline install + load model + track a synthetic 3D movie with the network hard-blocked.

Touches NO competition data - pure dependency/plumbing test.

In [ ]:
# ================= CONFIG =================
import os, glob, sys, subprocess
from importlib.metadata import version as pkgver

# 'download'     -> Phase 1, INTERNET ON:  fetch wheels + pretrained model into /kaggle/working
# 'offline_test' -> Phase 2, INTERNET OFF: install from wheels + load model + track a synthetic movie
PHASE = 'download'      # <-- flip to 'offline_test' for the second run (Internet OFF, dataset attached)

MODEL_NAME  = 'general_2d'   # pretrained model to bundle; Phase 1 also PROBES for other (esp. 3D) models
WORK        = '/kaggle/working'
WHEELS_DIR  = os.path.join(WORK, 'trackastra_wheels')
MODEL_DIR   = os.path.join(WORK, 'trackastra_models')
PRUNE_TORCH = True     # drop torch/nvidia wheels (Kaggle image already ships torch) to keep the dataset small

# Pin numpy/scipy to the versions ALREADY in the Kaggle image so pip never upgrades them (root cause of run-1 crash).
NP_VER = pkgver('numpy'); SP_VER = pkgver('scipy')
print('PHASE =', PHASE, '| python', sys.version.split()[0], '| base numpy', NP_VER, '| base scipy', SP_VER)

In [ ]:
# ===== subprocess scripts (run trackastra in a FRESH interpreter -> no stale-numpy issues) =====
FETCH_SRC = '''
import os, sys, shutil
import numpy, scipy
print('SUBPROC numpy', numpy.__version__, '| scipy', scipy.__version__)
import torch
from trackastra.model import Trackastra
name, model_dir = sys.argv[1], sys.argv[2]
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
try:
    Trackastra.from_pretrained(name, device=dev)
    print('from_pretrained OK:', name)
except Exception as e:
    print('from_pretrained FAILED:', repr(e)[:400])
# Copy the downloaded model registry into model_dir so Phase 2 can load it OFFLINE.
# Real location on Kaggle = platformdirs user_data_dir -> ~/.local/share/trackastra/models/<name>/
try:
    from platformdirs import user_data_dir
    base = user_data_dir('trackastra')
except Exception:
    base = os.path.expanduser('~/.local/share/trackastra')
copied = False
for cand in [base, os.path.expanduser('~/.local/share/trackastra'), os.path.expanduser('~/.cache/trackastra')]:
    mp = os.path.join(cand, 'models')
    if os.path.isdir(mp):
        shutil.copytree(mp, model_dir, dirs_exist_ok=True)
        print('copied models', mp, '->', model_dir)
        copied = True; break
if not copied:
    print('WARNING: no models dir found to bundle (looked under user_data_dir / ~/.local/share / ~/.cache)')
for root, dirs, files in os.walk(model_dir):
    for fn in files:
        print('  bundled:', os.path.join(root, fn))
'''

TEST_SRC = '''
import os, sys, socket, shutil
orig = socket.socket.connect
def blocked(self, *a, **k):
    raise RuntimeError('NETWORK BLOCKED: a connect was attempted -> not offline-safe')
socket.socket.connect = blocked
print('network hard-blocked in subprocess')
import numpy as np, torch
print('SUBPROC numpy', np.__version__)
from trackastra.model import Trackastra
model_root, name = sys.argv[1], sys.argv[2]
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
# seed trackastra's cache from the bundled model so from_pretrained can load with NO download
try:
    from platformdirs import user_data_dir
    cache_models = os.path.join(user_data_dir('trackastra'), 'models')
    os.makedirs(cache_models, exist_ok=True)
    dst = os.path.join(cache_models, os.path.basename(model_root.rstrip('/')))
    if os.path.isdir(model_root) and not os.path.isdir(dst):
        shutil.copytree(model_root, dst)
    print('seeded cache ->', dst)
except Exception as e:
    print('cache seed skipped:', repr(e)[:200])
model = None
for loader in ('from_folder', 'from_pretrained'):
    try:
        model = Trackastra.from_folder(model_root, device=dev) if loader == 'from_folder' else Trackastra.from_pretrained(name, device=dev)
        print('loaded via', loader); break
    except Exception as e:
        print(loader, 'FAILED:', repr(e)[:300])
assert model is not None, 'could not load a bundled model offline'
T, Z, Y, X = 6, 16, 64, 64
rng = np.random.default_rng(0); n = 5
starts = rng.uniform([2, 6, 6], [Z-2, X-6, X-6], size=(n, 3))
vels = rng.uniform(-1, 1, size=(n, 3))
imgs = np.zeros((T, Z, Y, X), np.float32); masks = np.zeros((T, Z, Y, X), np.uint16)
zz, yy, xx = np.ogrid[:Z, :Y, :X]
for t in range(T):
    for i in range(n):
        cz, cy, cx = starts[i] + vels[i] * t
        ball = ((zz-cz)**2/4.0 + (yy-cy)**2 + (xx-cx)**2) <= 6.0
        imgs[t][ball] = 1.0; masks[t][ball] = i + 1
print('synthetic', imgs.shape)
try:
    tg = model.track(imgs, masks, mode='greedy')
    g = getattr(tg, 'graph', tg)
    try:
        print('TRACK OK nodes', g.number_of_nodes(), 'edges', g.number_of_edges())
    except Exception:
        print('TRACK OK', type(tg))
    print('=== OFFLINE GATE PASSED ===')
except Exception as e:
    print('track FAILED:', repr(e)[:400])
    print('NOTE: a 2D-only model on 3D data fails here -> no 3D pretrained -> train our own.')
finally:
    socket.socket.connect = orig
'''
print('subprocess script sources ready')

In [ ]:
# ============ PHASE 1 (INTERNET ON): download wheels + fetch pretrained model ============
def sh(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-1800:]);  print('ERR:', r.stderr[-1200:]);  return r

if PHASE == 'download':
    os.makedirs(WHEELS_DIR, exist_ok=True); os.makedirs(MODEL_DIR, exist_ok=True)

    # 1) download the full wheel closure (installable offline later)
    print('downloading trackastra wheels ...')
    sh([sys.executable, '-m', 'pip', 'download', 'trackastra', '-d', WHEELS_DIR])

    # 2) install trackastra with numpy/scipy PINNED so the resolver does NOT upgrade them (run-1 crash fix)
    print('installing trackastra (numpy/scipy pinned to Kaggle versions) ...')
    sh([sys.executable, '-m', 'pip', 'install', '-q', 'trackastra', f'numpy=={NP_VER}', f'scipy=={SP_VER}'])

    # 3) fetch the pretrained model + probe available models in a FRESH SUBPROCESS.
    #    Helper script goes to /tmp (NOT /kaggle/working) so it never lands in the saved dataset.
    fetch = '/tmp/_tra_fetch.py'
    with open(fetch, 'w') as f: f.write(FETCH_SRC)
    print('--- model-fetch subprocess ---')
    r = subprocess.run([sys.executable, fetch, MODEL_NAME, MODEL_DIR], capture_output=True, text=True)
    print(r.stdout[-3000:]); print('ERR:', r.stderr[-3000:])

    # 4) keep the output CLEAN: drop any stray helper scripts a prior run may have left in /kaggle/working
    #    so the dataset = wheels + models only (no *.py).
    for junk in glob.glob(os.path.join(WORK, '_tra_*.py')):
        os.remove(junk); print('removed stray', junk)

    # 5) prune torch/nvidia wheels (Kaggle ships torch) to shrink the dataset
    if PRUNE_TORCH:
        big = ('torch','nvidia','triton','cusparse','cudnn','cublas','cufft','curand','cusolver','nccl','cuda')
        for w in glob.glob(os.path.join(WHEELS_DIR, '*.whl')):
            if os.path.basename(w).lower().startswith(big): os.remove(w)

    wheels = sorted(glob.glob(os.path.join(WHEELS_DIR, '*.whl')))
    print(); print('WHEELS:', len(wheels), 'files,', round(sum(os.path.getsize(w) for w in wheels)/1e6, 1), 'MB')
    print('MODEL files:')
    for fp in glob.glob(os.path.join(MODEL_DIR, '**', '*'), recursive=True):
        if os.path.isfile(fp): print('  ', fp, round(os.path.getsize(fp)/1e6, 2), 'MB')
    print(); print('OUTPUT is now wheels + models only (helper scripts live in /tmp).')
    print('NEXT: Save Version -> make a Kaggle dataset from /kaggle/working (e.g. trackastra-offline),')
    print('      then re-run with PHASE=offline_test + Internet OFF + that dataset attached.')
else:
    print('PHASE != download -> skipping Phase 1')

In [ ]:
# ===== PROBE (Phase 1, INTERNET ON): enumerate ALL pretrained models + test 2D vs 3D support =====
# Answers the open question: is there a usable 3D pretrained model, or must we TRAIN our own?
# Run with PHASE='download' + Internet ON, AFTER the install cell (trackastra must be installed). Downloads
# each valid candidate model (network) and tries a tiny 2D and 3D track on it. Runs in a fresh subprocess.
PROBE_SRC = '''
import inspect, numpy as np, torch
from trackastra.model import Trackastra
from trackastra.model import pretrained
print('=== source of trackastra.model.pretrained (reveals real model names / download URLs) ===')
try:
    print(inspect.getsource(pretrained))
except Exception as e:
    print('getsource failed:', repr(e)[:200])
print('=== bogus-name error (libraries often list the valid names here) ===')
try:
    Trackastra.from_pretrained('__does_not_exist__', device='cpu')
except Exception as e:
    print(repr(e)[:800])

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
def tiny(nd):
    if nd == 2:
        T, Y, X = 4, 48, 48
        imgs = np.zeros((T, Y, X), np.float32); masks = np.zeros((T, Y, X), np.uint16)
        yy, xx = np.ogrid[:Y, :X]
        for t in range(T):
            for i in range(3):
                cy, cx = 8 + 8 * i + t, 10 + 8 * i
                b = ((yy - cy) ** 2 + (xx - cx) ** 2) <= 4
                imgs[t][b] = 1.0; masks[t][b] = i + 1
        return imgs, masks
    T, Z, Y, X = 4, 12, 48, 48
    imgs = np.zeros((T, Z, Y, X), np.float32); masks = np.zeros((T, Z, Y, X), np.uint16)
    zz, yy, xx = np.ogrid[:Z, :Y, :X]
    for t in range(T):
        for i in range(3):
            cz, cy, cx = 6, 8 + 8 * i + t, 10 + 8 * i
            b = ((zz - cz) ** 2 + (yy - cy) ** 2 + (xx - cx) ** 2) <= 4
            imgs[t][b] = 1.0; masks[t][b] = i + 1
    return imgs, masks

candidates = ['general_2d', 'ctc', 'general_3d', 'ctc_3d', '3d', 'general', 'bacteria_2d',
              'fluo_3d', 'trackastra_3d', 'ctc_2d', 'general_2d_3d']
print('=== per-candidate: does it load, and does it accept 2D / 3D input? ===')
for name in candidates:
    try:
        m = Trackastra.from_pretrained(name, device=dev)
    except Exception as e:
        print('  ', name, '-> NOT available:', repr(e)[:110]); continue
    res = []
    for nd in (2, 3):
        imgs, masks = tiny(nd)
        try:
            m.track(imgs, masks, mode='greedy'); res.append(str(nd) + 'D:OK')
        except Exception as e:
            res.append(str(nd) + 'D:' + repr(e)[:55])
    print('  ', name, '->', ' | '.join(res))
'''
if PHASE == 'download':
    pp = '/tmp/_tra_probe.py'
    with open(pp, 'w') as f: f.write(PROBE_SRC)
    print('--- model probe subprocess (Internet ON) ---')
    r = subprocess.run([sys.executable, pp], capture_output=True, text=True)
    print(r.stdout[-15000:]); print('ERR:', r.stderr[-2000:])
    for junk in glob.glob(os.path.join(WORK, '_tra_*.py')):
        os.remove(junk)
else:
    print('PROBE runs only in PHASE=download (Internet ON)')

In [ ]:
# ============ PHASE 2 (INTERNET OFF): offline install from wheels ============
def find_dir(patterns, roots=('/kaggle/input', '/kaggle/working')):
    for root in roots:
        for pat in patterns:
            hits = glob.glob(os.path.join(root, '**', pat), recursive=True)
            if hits: return os.path.dirname(hits[0])
    return None

if PHASE == 'offline_test':
    wheeldir = find_dir(['trackastra*.whl'])
    assert wheeldir, 'no trackastra wheels under /kaggle/input or /kaggle/working -- attach the Phase-1 dataset'
    # UNPINNED on purpose: base numpy 2.0.2 conflicts with imagecodecs>=2.1 (the only imagecodecs wheel available).
    # trackastra runs in a fresh SUBPROCESS in the next cell, so pip upgrading numpy in this kernel is harmless.
    print('installing offline from', wheeldir)
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', wheeldir, 'trackastra'],
                       capture_output=True, text=True)
    print(r.stdout[-4000:]); print('ERR:', r.stderr[-2500:])
    print('offline install returncode:', r.returncode, '(0 = OK)')
    if r.returncode != 0:
        print('WARNING: offline install failed (see ERR) -- a dep may be missing from the wheels.')
else:
    print('PHASE != offline_test -> skipping Phase 2 install')

In [ ]:
# ====== PHASE 2: load model + track a synthetic 3D movie in a FRESH SUBPROCESS (network hard-blocked) ======
if PHASE == 'offline_test':
    model_root = (find_dir(['*.pt', '*.pth', 'config.yaml', 'config.json']) or MODEL_DIR)
    print('model_root guess:', model_root)
    test = '/tmp/_tra_test.py'   # helper in /tmp so it never lands in /kaggle/working
    with open(test, 'w') as f: f.write(TEST_SRC)
    r = subprocess.run([sys.executable, test, model_root, MODEL_NAME], capture_output=True, text=True)
    print('--- offline test subprocess ---'); print(r.stdout[-4000:]); print('ERR:', r.stderr[-3000:])
    print(); print('>>> GATE PASSED <<<' if 'OFFLINE GATE PASSED' in r.stdout
                   else '>>> GATE NOT passed -- read the subprocess output/ERR above <<<')
else:
    print('PHASE != offline_test -> skipping Phase 2 test')

## How to read this

**PASS** = Phase 2 prints `>>> GATE PASSED <<<` (installed offline + loaded model + tracked, with the subprocess network hard-blocked). Note **(i)** the wheels dataset size and **(ii)** whether a 3D pretrained model exists (Phase 1 `pretrained names:` probe + whether the synthetic 3D track ran).

**Failure modes - each is a real finding:**
- *Phase 1 pip install fails on the numpy/scipy pin* (trackastra needs a newer numpy than Kaggle ships): note the required range; fallback is to let it upgrade numpy AND `pip install --force-reinstall numpy==<ver>` so python+C match, still running trackastra in a subprocess.
- *Offline pip install fails (missing dep):* add it in Phase 1 (`pip download <dep> -d WHEELS_DIR`); or a compiled/platform wheel mismatch -> record it.
- *No 3D pretrained model (only general_2d etc.):* zero-shot 3D is OFF the table -> Phase 2 = train Trackastra on our 199 pairs (we have GT edges + divisions). Still viable, not free.
- *greedy pulls an ILP/solver import (motile/ilpy):* record which solver -> packaging is harder.
- *Both `from_folder` and `from_pretrained` fail:* inspect the `model_root` contents and adjust the loader (API may differ across versions).

**If PASS:** next = (1) read pilkwang's learned-graph notebook; (2) reuse the centroids->pseudo-mask step to feed a REAL train sample through Trackastra and compare local edge-Jaccard vs the v2 two-pass linker (0.859 local).